# 🥁 HT-Demucs FT: Drums Specialist Playground

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shinohara-rin/stuff/blob/main/htdemucs_drums_playground.ipynb)

A hands-on playground for [`StemSplitio/htdemucs-ft-drums-pytorch`](https://huggingface.co/StemSplitio/htdemucs-ft-drums-pytorch):
**sub-model 0 of Meta's 4-model `htdemucs_ft` bag**, the one trained to pull out **drums**.
It's about a quarter of the full ensemble's size and compute, and its drums output is the same (median SDR ≈ 10.1 dB on MUSDB18-HQ).

**What you can do here**
1. Load the drums specialist (on a GPU if one is available)
2. Feed it audio: a built-in demo track, your own upload, a direct URL, or YouTube
3. Separate it into **drums** and **everything else**, and listen to both
4. Look at waveforms and spectrograms
5. Remix the drum level up or down
6. Analyse the drum stem: tempo, beats, onsets, a click track, and auto-sliced one-shots
7. Download everything as a zip
8. *(Optional)* Compare against the full 4-model ensemble

> 💡 **Tip:** In Colab, go to *Runtime → Change runtime type → T4 GPU*. A 3-minute song takes seconds on a GPU and around a minute on CPU.

---

## 1 · Setup

In [ ]:
# Demucs 4.0.1 is the version the model card / handler.py targets.
!pip install -q demucs==4.0.1 yt-dlp

In [ ]:
import os, io, time, shutil, subprocess, zipfile, urllib.request
import numpy as np
import torch
import soundfile as sf
import librosa, librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display, Markdown

from demucs.pretrained import get_model
from demucs.apply import apply_model
from demucs.audio import convert_audio

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")
print(f"torch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""))
if device == "cpu":
    print("⚠️  No GPU found. This still works, just slower. Runtime → Change runtime type → GPU.")

WORK = "/content" if IN_COLAB else os.path.abspath(".")
IN_DIR, OUT_DIR = os.path.join(WORK, "inputs"), os.path.join(WORK, "outputs")
os.makedirs(IN_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

## 2 · Load the drums specialist

The HF repo doesn't ship separate weights. Its `handler.py` loads the official `htdemucs_ft` bag through Demucs and keeps only `bag.models[0]`. We do the same here. The first run downloads about 320 MB of checkpoints from Meta's CDN.

In [ ]:
import hashlib, yaml
from demucs.pretrained import REMOTE_ROOT, _parse_remote_files

def prefetch_checkpoints(bag_name="htdemucs_ft"):
    # Download the bag's checkpoints into the torch.hub cache ourselves, verifying the
    # sha256 prefix in the filename and retrying on a bad download (torch.hub would
    # just raise "invalid hash value"). Demucs then loads them from the cache.
    urls = _parse_remote_files(REMOTE_ROOT / "files.txt")
    with open(REMOTE_ROOT / f"{bag_name}.yaml") as f:
        sigs = yaml.safe_load(f)["models"]
    ckpt_dir = os.path.join(torch.hub.get_dir(), "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)
    for sig in sigs:
        url = urls[sig]
        fname = url.rsplit("/", 1)[1]                  # e.g. f7e0c4bc-ba3fe64a.th
        dst = os.path.join(ckpt_dir, fname)
        if os.path.exists(dst):
            continue
        tmp = dst + ".partial"
        expected = fname.rsplit(".", 1)[0].split("-")[-1]
        for attempt in range(3):
            torch.hub.download_url_to_file(url, tmp, progress=True)
            with open(tmp, "rb") as f:
                digest = hashlib.sha256(f.read()).hexdigest()
            if digest.startswith(expected):
                os.replace(tmp, dst)
                break
            print(f"Checksum mismatch for {fname} (attempt {attempt+1}), retrying…")
        else:
            os.remove(tmp)
            raise RuntimeError(f"Could not download a valid {fname}")

t0 = time.time()
prefetch_checkpoints("htdemucs_ft")
bag = get_model("htdemucs_ft")         # 4-model bag; each sub-model specialises in one stem
model = bag.models[0].eval().to(device)  # sub-model 0 = drums specialist

SR       = int(bag.samplerate)           # 44100
CHANNELS = int(bag.audio_channels)       # 2
SOURCES  = list(bag.sources)             # ['drums', 'bass', 'other', 'vocals']
DRUMS    = SOURCES.index("drums")

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Params: {n_params/1e6:.1f} M  (~{n_params*4/2**20:.0f} MB fp32)")
print(f"Sample rate: {SR} Hz | channels: {CHANNELS} | sources: {SOURCES}")
print(f"Bag weights for sub-model 0: {dict(zip(SOURCES, bag.weights[0]))}")
print(f"Native segment length: {float(model.segment):.2f}s")

## 3 · Pick your audio

Choose a `SOURCE`:
- **demo**: royalty-free tracks from the [librosa example set](https://librosa.org/doc/latest/recordings.html) (CC-BY / CC0)
- **upload**: a file from your computer (mp3, wav, flac, ogg, m4a, …)
- **url**: a direct link to an audio file
- **youtube**: any URL `yt-dlp` can handle. Only use audio you have the rights to.

`START_S` / `DURATION_S` trim the input so experiments stay quick. Set `DURATION_S = 0` to process the whole file.

In [ ]:
SOURCE     = "demo"     # @param ["demo", "upload", "url", "youtube"]
DEMO_TRACK = "vibeace"  # @param ["vibeace", "fishin", "choice", "sweetwaltz"]
URL        = ""         # @param {type:"string"}
START_S    = 0          # @param {type:"number"}
DURATION_S = 30         # @param {type:"number"}


def fetch_input():
    if SOURCE == "demo":
        return librosa.ex(DEMO_TRACK)
    if SOURCE == "upload":
        if not IN_COLAB:
            raise RuntimeError("Upload widget is Colab-only; put a file next to the notebook and use SOURCE='url' with a file:// path.")
        up = colab_files.upload()
        name = next(iter(up))
        path = os.path.join(IN_DIR, os.path.basename(name))
        with open(path, "wb") as f:
            f.write(up[name])
        return path
    if not URL:
        raise ValueError("Set URL for SOURCE='url' / 'youtube'.")
    if SOURCE == "url":
        path = os.path.join(IN_DIR, os.path.basename(URL.split("?")[0]) or "download")
        urllib.request.urlretrieve(URL, path)
        return path
    if SOURCE == "youtube":
        import yt_dlp
        opts = {"format": "bestaudio/best", "outtmpl": os.path.join(IN_DIR, "yt.%(ext)s"),
                "quiet": True, "noplaylist": True}
        with yt_dlp.YoutubeDL(opts) as ydl:
            info = ydl.extract_info(URL, download=True)
            print("Downloaded:", info.get("title"))
            return ydl.prepare_filename(info)
    raise ValueError(SOURCE)


def load_audio(path, start=0.0, duration=0.0):
    # Decode with ffmpeg (handles almost any format), trim, and resample to the model rate.
    cmd = ["ffmpeg", "-v", "error", "-ss", str(start), "-i", path]
    if duration and duration > 0:
        cmd += ["-t", str(duration)]
    cmd += ["-f", "f32le", "-acodec", "pcm_f32le", "-ac", str(CHANNELS), "-ar", str(SR), "-"]
    raw = subprocess.run(cmd, check=True, capture_output=True).stdout
    wav = np.frombuffer(raw, dtype=np.float32).reshape(-1, CHANNELS).T.copy()
    return torch.from_numpy(wav)  # (channels, samples)


in_path = fetch_input()
mix = load_audio(in_path, START_S, DURATION_S)
if mix.shape[-1] == 0:
    raise ValueError("Got no audio. Check START_S / DURATION_S against the track length.")
TRACK_NAME = os.path.splitext(os.path.basename(in_path))[0].replace(" ", "_")
print(f"{TRACK_NAME}: {mix.shape[-1]/SR:.1f}s, {mix.shape[0]} ch @ {SR} Hz")
display(Audio(mix.numpy(), rate=SR))

## 4 · Separate

Knobs (all standard `demucs.apply.apply_model` options):
- **`SHIFTS`**: random time-shift augmentation, averaged. Each shift adds one more full pass. Usually worth +0.1–0.3 dB SDR; `0` is fastest.
- **`OVERLAP`**: overlap between chunks. Higher means smoother joins and slower runs.
- **`SPLIT`**: process in chunks of the model's native segment length (keep this on for long tracks or low VRAM).

In [ ]:
SHIFTS  = 1     # @param {type:"slider", min:0, max:5, step:1}
OVERLAP = 0.25  # @param {type:"slider", min:0.0, max:0.75, step:0.05}
SPLIT   = True  # @param {type:"boolean"}

# Demucs normalises the input itself during training; matching that at inference
# (as demucs.separate does) helps on quiet/loud tracks.
ref = mix.mean(0)
mean, std = ref.mean(), ref.std() + 1e-8
x = ((mix - mean) / std).unsqueeze(0)

if device == "cuda":
    torch.cuda.synchronize()
t0 = time.time()
with torch.inference_mode():
    stems = apply_model(model, x, device=device, shifts=SHIFTS, split=SPLIT,
                        overlap=OVERLAP, progress=True)[0]
if device == "cuda":
    torch.cuda.synchronize()
elapsed = time.time() - t0
stems = (stems * std + mean).cpu()

drums = stems[DRUMS]       # the stem this specialist is trained for
no_drums = mix - drums     # residual = everything except drums
# Note: rows other than DRUMS in `stems` are weak by-products. The bag gives them weight 0.

dur = mix.shape[-1] / SR
print(f"Separated {dur:.1f}s of audio in {elapsed:.1f}s → RTF {elapsed/dur:.3f} "
      f"({dur/elapsed:.1f}× real-time) on {device}")

## 5 · Listen

In [ ]:
def player(label, wav):
    display(Markdown(f"**{label}**"))
    display(Audio(wav.numpy(), rate=SR, normalize=False if wav.abs().max() <= 1 else True))

player("Original mix", mix)
player("🥁 Drums (isolated)", drums)
player("🎸 Everything except drums (mix − drums)", no_drums)

## 6 · Visualise

In [ ]:
VIEW_START_S = 0   # @param {type:"number"}
VIEW_LEN_S   = 10  # @param {type:"number"}

a, b = int(VIEW_START_S * SR), int((VIEW_START_S + VIEW_LEN_S) * SR)
sigs = {"Mix": mix, "Drums": drums, "No drums": no_drums}
fig, axes = plt.subplots(2, 3, figsize=(18, 7), sharex="col")
for i, (name, w) in enumerate(sigs.items()):
    seg = w.mean(0)[a:b].numpy()
    t = np.arange(len(seg)) / SR + VIEW_START_S
    axes[0, i].plot(t, seg, lw=0.4)
    axes[0, i].set_ylim(-1, 1); axes[0, i].set_title(name)
    S = librosa.amplitude_to_db(np.abs(librosa.stft(seg, n_fft=2048, hop_length=512)), ref=np.max)
    librosa.display.specshow(S, sr=SR, hop_length=512, x_axis="time", y_axis="log",
                             ax=axes[1, i], vmin=-80, vmax=0)
    axes[1, i].set_xlabel("time (s, from view start)")
axes[0, 0].set_ylabel("amplitude")
plt.tight_layout(); plt.show()

def rms_db(w):
    return 20 * np.log10(w.pow(2).mean().sqrt().item() + 1e-12)
print(f"RMS  mix {rms_db(mix):.1f} dBFS | drums {rms_db(drums):.1f} dBFS | no-drums {rms_db(no_drums):.1f} dBFS")

## 7 · Remix playground

Turn the drums up, down, or off. `-60` effectively mutes them; `+6` gives a drum-forward mix. Re-run the cell after you move the sliders.

In [ ]:
DRUM_GAIN_DB  = 6   # @param {type:"slider", min:-60, max:12, step:1}
OTHER_GAIN_DB = 0   # @param {type:"slider", min:-60, max:12, step:1}

g_d, g_o = 10 ** (DRUM_GAIN_DB / 20), 10 ** (OTHER_GAIN_DB / 20)
remix = g_d * drums + g_o * no_drums
peak = remix.abs().max().item()
if peak > 1:
    remix = remix / peak * 0.99
    print(f"(peak {peak:.2f} > 1, normalised to avoid clipping)")
player(f"Remix: drums {DRUM_GAIN_DB:+d} dB, rest {OTHER_GAIN_DB:+d} dB", remix)

## 8 · Drum analysis: tempo, beats, onsets, one-shots

One of the model's main uses is feeding a clean drum stem into beat and onset detection. Here we compare beat tracking on the **mix** and on the **isolated drums**, render a click track over the drums, and auto-slice the strongest hits into one-shot samples.

In [ ]:
N_ONESHOTS   = 8    # @param {type:"slider", min:1, max:32, step:1}
ONESHOT_MS   = 250  # @param {type:"slider", min:50, max:1000, step:10}

ASR = 22050  # analysis rate
d_mono = librosa.resample(drums.mean(0).numpy(), orig_sr=SR, target_sr=ASR)
m_mono = librosa.resample(mix.mean(0).numpy(),   orig_sr=SR, target_sr=ASR)

for name, y in [("mix", m_mono), ("drums", d_mono)]:
    tempo, _ = librosa.beat.beat_track(y=y, sr=ASR)
    print(f"Tempo estimate from {name:5s}: {float(np.atleast_1d(tempo)[0]):6.1f} BPM")

env = librosa.onset.onset_strength(y=d_mono, sr=ASR)
tempo, beats = librosa.beat.beat_track(onset_envelope=env, sr=ASR)
onsets = librosa.onset.onset_detect(onset_envelope=env, sr=ASR, backtrack=True)
beat_t, onset_t = librosa.frames_to_time(beats, sr=ASR), librosa.frames_to_time(onsets, sr=ASR)
print(f"{len(beat_t)} beats, {len(onset_t)} drum onsets detected")

fig, ax = plt.subplots(figsize=(18, 3))
t_env = librosa.times_like(env, sr=ASR)
ax.plot(t_env, env / env.max(), lw=0.8, label="onset strength (drums)")
ax.vlines(beat_t, 0, 1, color="r", alpha=0.5, lw=1, label="beats")
ax.vlines(onset_t, 0, 0.3, color="k", alpha=0.6, lw=0.8, label="onsets")
ax.set_xlim(VIEW_START_S, VIEW_START_S + VIEW_LEN_S); ax.set_xlabel("time (s)"); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

clicks = librosa.clicks(times=beat_t, sr=ASR, length=len(d_mono), click_freq=1500)
display(Markdown("**Drums + beat click track**"))
display(Audio(np.stack([d_mono, d_mono]) * 0.8 + clicks * 0.3, rate=ASR))

# One-shots: take the strongest onsets, keeping each slice from overlapping the next one.
slice_len = int(ONESHOT_MS / 1000 * SR)
d_full = drums.numpy()
onset_samples = (onset_t * SR).astype(int)
strength = env[onsets] if len(onsets) else np.array([])
picked, taken = [], np.zeros(d_full.shape[-1], bool)
for idx in np.argsort(-strength):
    s = onset_samples[idx]
    if s + slice_len <= d_full.shape[-1] and not taken[s:s + slice_len].any():
        picked.append(s); taken[s:s + slice_len] = True
    if len(picked) >= N_ONESHOTS:
        break

fade = np.ones(slice_len, np.float32)
fl = min(slice_len // 4, int(0.02 * SR)); fade[-fl:] = np.linspace(1, 0, fl)
ONESHOTS = []
for s in sorted(picked):
    hit = d_full[:, s:s + slice_len] * fade
    ONESHOTS.append((s / SR, hit / (np.abs(hit).max() + 1e-9) * 0.9))

display(Markdown(f"**{len(ONESHOTS)} auto-sliced one-shots** (peak-normalised)"))
if ONESHOTS:
    gap = np.zeros((CHANNELS, int(0.15 * SR)), np.float32)
    display(Audio(np.concatenate(sum([[h, gap] for _, h in ONESHOTS], []), axis=1), rate=SR))
    print("at:", ", ".join(f"{t:.2f}s" for t, _ in ONESHOTS))

## 9 · Save & download

In [ ]:
SUBTYPE = "PCM_16"  # @param ["PCM_16", "PCM_24", "FLOAT"]

def save(name, wav, rate=SR):
    arr = wav.numpy() if torch.is_tensor(wav) else wav
    if SUBTYPE != "FLOAT":
        arr = np.clip(arr, -1, 1)
    path = os.path.join(OUT_DIR, name)
    sf.write(path, arr.T, rate, subtype=SUBTYPE)
    return path

paths = [save(f"{TRACK_NAME}_drums.wav", drums),
         save(f"{TRACK_NAME}_no_drums.wav", no_drums),
         save(f"{TRACK_NAME}_remix.wav", remix)]
os.makedirs(os.path.join(OUT_DIR, "oneshots"), exist_ok=True)
for i, (t, hit) in enumerate(ONESHOTS):
    paths.append(save(f"oneshots/{TRACK_NAME}_hit{i:02d}_{t:.2f}s.wav", hit))

zip_path = os.path.join(WORK, f"{TRACK_NAME}_drums_playground.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p, os.path.relpath(p, OUT_DIR))
print("\n".join(os.path.relpath(p, WORK) for p in paths))
print(f"→ {zip_path} ({os.path.getsize(zip_path)/2**20:.1f} MB)")
if IN_COLAB:
    colab_files.download(zip_path)

## 10 · *(Optional)* Compare with the full 4-model ensemble

The model card says the bag's drums output **is** this sub-model's output, because the bag weights drums 1.0 for model 0 and 0 for the others. This cell checks that by running the full `htdemucs_ft` bag (about 4× the compute) and comparing. It also gives you bass, vocals and other stems to play with. With `SHIFTS > 0` the random shifts differ between runs, so expect a tiny, nonzero difference.

In [ ]:
RUN_FULL_BAG = False  # @param {type:"boolean"}

if RUN_FULL_BAG:
    t0 = time.time()
    with torch.inference_mode():
        full = apply_model(bag, x, device=device, shifts=0, split=SPLIT,
                           overlap=OVERLAP, progress=True)[0]
        solo = apply_model(model, x, device=device, shifts=0, split=SPLIT,
                           overlap=OVERLAP, progress=False)[0]
    full, solo = (full * std + mean).cpu(), (solo * std + mean).cpu()
    print(f"Full bag: {time.time()-t0:.1f}s (includes a shifts=0 specialist re-run)")
    diff = (full[DRUMS] - solo[DRUMS]).abs().max().item()
    print(f"max |bag drums − specialist drums| = {diff:.2e}")
    for i, s in enumerate(SOURCES):
        player(f"Full bag: {s}", full[i])
else:
    print("Set RUN_FULL_BAG = True and re-run to compare.")

---
### Notes
- **Credits:** HT-Demucs by Rouard, Massa & Défossez (Meta AI, ICASSP 2023), MIT license, [facebookresearch/demucs](https://github.com/facebookresearch/demucs). Specialist packaging by [StemSplit](https://huggingface.co/StemSplitio).
- **Memory:** with `SPLIT=True`, a full-length song fits easily on a free T4. If you hit OOM, lower `OVERLAP` or trim with `DURATION_S`.
- **Other specialists** (vocals / bass / other) are `bag.models[1..3]`. Check `bag.weights` to see which sub-model owns which stem.